# Lab 4.2 — Multi-Agent Sales Workflow with Tools & Handoffs

## Introduction

This is the **second lab** in the OpenAI Agents SDK module, building on [Lab 4.1](4_lab1.ipynb). You will build a **multi-agent sales workflow** for a fictional company called **ComplAI** (SOC2 compliance SaaS).

The lab progresses through three major patterns:
1. **Multi-agent workflow** — three sales agents with different writing styles, run in parallel, with a picker agent to choose the best email
2. **Tools** — wrap Python functions and other agents as tools using `@function_tool` and `.as_tool()`, orchestrated by a Sales Manager agent
3. **Handoffs** — delegate control to another agent (Email Manager) for formatting and sending HTML email

This lab replaces the manual JSON tool schemas and `handle_tool_calls()` loop from Lab 3.2 with the Agents SDK's built-in tool handling.

Before starting, make sure you have:
- Completed Lab 4.1 and Lab 3.2 (MailerSend setup)
- `OPENAI_API_KEY`, `YOUR_EMAIL`, and `YOUR_NAME` in your `.env` file
- `MAILER_SEND_API_TOKEN` and `MAILER_SEND_DOMAIN` configured (see Lab 3.2 for MailerSend setup steps)

---

## Intention (Learning Objectives)

By the end of this lab, you should be able to:

1. **Create multiple specialized agents** — define agents with different instructions for the same task
2. **Stream and parallelize agent runs** — use `Runner.run_streamed()` and `asyncio.gather()` with `Runner.run()`
3. **Build a judge agent** — have one agent evaluate outputs from other agents
4. **Define tools with `@function_tool`** — convert Python functions into tools without manual JSON schemas
5. **Convert agents to tools** — use `.as_tool()` so one agent can call another
6. **Orchestrate with a planning agent** — give a manager agent tools and step-by-step instructions
7. **Use handoffs** — pass control from one agent to another (vs. tools, where control returns)

---

## Lab Structure

| Part | Content |
|------|---------|
| **Part 1** | Setup — imports, environment variables, MailerSend credentials |
| **Part 2** | Multi-agent workflow — 3 sales agents, streaming, parallel runs, picker |
| **Part 3** | Tools — `@function_tool`, agents as tools, Sales Manager orchestration |
| **Part 4** | Handoffs — Email Manager agent, HTML email pipeline, full automated SDR |

> **Note:** Run cells top to bottom in order. Check [OpenAI Traces](https://platform.openai.com/traces) and your inbox after each major run.

## Part 1 — Setup

Import dependencies, load `.env`, and verify your personal and MailerSend credentials.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import os
from mailersend import MailerSendClient, EmailBuilder
import asyncio

In [2]:
load_dotenv(override=True)

True

In [ ]:
# For your data
your_email = os.getenv("YOUR_EMAIL")
your_name = os.getenv("YOUR_NAME")

if your_email:
    print(f"Your Email found and starts with {your_email[0:5]}")
else:
    print("Your Email not found")

if your_name:
    print(f"Your Name found and starts with {your_name[0:5]}")
else:
    print("Your Name not found")

Your Email found and starts with b5923
Your Name found and starts with Aphir


In [4]:
# For mail sender

mailer_send_api_token = os.getenv("MAILER_SEND_API_TOKEN")
mailer_send_domain = os.getenv("MAILER_SEND_DOMAIN")

if mailer_send_api_token:
    print(f"Mailer Send API Token found and starts with {mailer_send_api_token[0:5]}")
else:
    print("Mailer Send API Token not found")

if mailer_send_domain:
    print(f"Mailer Send Domain found and starts with {mailer_send_domain[0:5]}")
else:
    print("Mailer Send Domain not found")

Mailer Send API Token found and starts with mlsn.
Mailer Send Domain found and starts with test-


## Part 2 — Multi-Agent Sales Workflow

Create three sales agents with different tones (professional, engaging, concise). Stream one agent's output, then run all three in parallel and use a **picker agent** to select the best cold email.

In [8]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [9]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini"
)

In [10]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC 2 Compliance with ComplAI

Hi [Recipient's Name],

I hope this message finds you well.

I’m reaching out to introduce you to ComplAI, a sophisticated SaaS tool designed to simplify the process of achieving and maintaining SOC 2 compliance. Our AI-powered platform not only streamlines compliance management but also prepares your organization for audits with exceptional efficiency.

With increasing regulatory demands and the need for transparent data practices, ensuring compliance can be both time-consuming and complex. ComplAI automates critical tasks, provides real-time insights, and helps identify gaps in your current processes, ultimately saving your team valuable time and resources.

If you’re looking to enhance your compliance efforts while minimizing risks, I’d love to schedule a brief call to discuss how ComplAI can help your organization thrive in today's regulatory landscape.

Thank you for considering this opportunity. I look forward to the possibi

In [11]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Streamline Your SOC 2 Compliance with AI-Powered Solutions

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m reaching out from ComplAI, where we specialize in delivering AI-driven solutions for SOC 2 compliance and audit preparation.

Navigating the complexities of SOC 2 can be both time-consuming and challenging. Our platform automates key compliance tasks, enabling your team to focus on what matters most—your business. By integrating our tool, you can expect:

- **Reduced Compliance Costs:** Save time and resources with automated tracking and reporting.
- **Enhanced Audit Readiness:** Ensure that your documentation is always up to date, minimizing surprises during audits.
- **Real-Time Insights:** Utilize AI analytics for actionable compliance insights tailored to your specific needs.

I would love to schedule a brief call to discuss how ComplAI can help streamline your compliance processes and mitigate risks.

Are you available for

In [12]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

In [13]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
Subject: Your SOC2 Compliance Just Called — It Wants a Makeover! ✨

Hey [Recipient's Name],

I hope this email finds you compliance-ready and living your best audit-free life! 🌟

I’m reaching out because I couldn't help but notice you’re probably juggling more compliance tasks than a circus performer on a unicycle. 🤹‍♂️ But what if I told you that dealing with SOC2 compliance could feel like a breezy walk in the park instead?

At ComplAI, we’ve created a SaaS tool that’s like your overly prepared friend who always has your back—except this one is powered by AI! Imagine a world where you can confidently sail through audits, while your competitors are still deciphering the instructions for their compliance puzzle. 

Here’s what our tool can do for you:
- Simplify SOC2 requirements with an easy-to-follow checklist
- Automate documentation, so you can spend less time writing and more time beach-waving 🏖️
- Get expert insights to keep you ahead of the compliance game

Curi

### Check the trace

Review the parallel and selection runs on the OpenAI Traces dashboard: **https://platform.openai.com/traces**

## Part 3 — Tools & Sales Manager

Add tools to the workflow. The `@function_tool` decorator converts a Python function into a tool automatically — no manual JSON schema needed. You can also convert an agent into a tool with `.as_tool()`.

In [14]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini",
)

### Inspect an agent

Run the cell below to see the agent's configuration, including its empty `tools` list before we add any.

In [15]:
sales_agent1

Agent(name='Professional Sales Agent', instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoff_description=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

### Create a `send_email` tool

Wrap the MailerSend helper with `@function_tool`. The SDK generates the JSON schema for you.

In [26]:
ms = MailerSendClient(api_key=mailer_send_api_token)

@function_tool
def send_email(sub: str, message: str):
    email = (EmailBuilder()
         .from_email(f"sales@{mailer_send_domain}", "Sales Agent")
         .to_many([{"email": your_email, "name": your_name }])
         .subject(sub)
         .text(message)
         .build())

    response = ms.emails.send(email)
    print(f"Email sent: {response.id}")

### Inspect the generated tool

Notice how `send_email` is now a `FunctionTool` with an auto-generated `params_json_schema`.

In [27]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='', params_json_schema={'properties': {'sub': {'title': 'Sub', 'type': 'string'}, 'message': {'title': 'Message', 'type': 'string'}}, 'required': ['sub', 'message'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112fb6980>, strict_json_schema=True, is_enabled=True)

### Convert an agent to a tool

Any agent can become a callable tool for another agent using `.as_tool()`.

In [28]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112fb7920>, strict_json_schema=True, is_enabled=True)

### Assemble all tools

Combine the three sales-agent tools and the `send_email` tool into a single list.

In [29]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112fb6160>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112fb71a0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required'

### Sales Manager — planning agent with tools

The Sales Manager agent receives step-by-step instructions: generate three drafts using the sales tools, pick the best one, then send it via `send_email`. Run it inside a trace and check your inbox.

In [ ]:
# Improved instructions thanks to student Guillermo F.

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool (which requires a subject and message) — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

Email sent: 6a32a0c14f85757dfa1b9262


### Check the trace & your email

**https://platform.openai.com/traces** — then verify the email arrived in your inbox.

## Part 4 — Handoffs

**Handoffs vs. tools:** with tools, control passes back to the calling agent after execution. With handoffs, control passes *across* to another agent entirely.

Build an **Email Manager** agent that writes a subject line, converts the body to HTML, and sends the email. The Sales Manager will hand off the winning draft instead of sending directly.

In [32]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [33]:
@function_tool
def send_email_html(subject: str, html_body: str)-> Dict[str, str]:
    email = (EmailBuilder()
         .from_email(f"sales@{mailer_send_domain}", "Sales Agent")
         .to_many([{"email": your_email, "name": your_name }])
         .subject(subject)
         .html(html_body)
         .build())

    response = ms.emails.send(email)
    print(f"Email sent: {response.id}")
    return {"status": "success"}

In [34]:
tools = [subject_tool, html_tool, send_email_html]

In [35]:
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1130f8fe0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1130f8f40>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='send_email_html', description='', params_json_schema={'properties': {'subject': {'title': 'Subject', 't

In [36]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")

In [37]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112fb6160>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112fb71a0>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': 

### Full automated SDR with handoffs

Reconfigure the Sales Manager with `handoffs=[emailer_agent]`. It generates drafts, picks the best, then hands off to the Email Manager for formatting and delivery.

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

---

## Next Steps

Review the full workflow on **https://platform.openai.com/traces** and check your email for the HTML message.

Experiment with: changing agent instructions, adding a fourth sales persona, or giving the Sales Manager permission to retry draft generation if quality is low.